#Transformed sprints data
- Read bronze sprints table
- Keep only the columns required for analytics (Drop url column)
- Standardise column names using snake_case (constructorId->constructor_id, driverId->driver_id, sprintName->sprint_name,positionText->finish_position_text)
- Rename columns(date->sprint_date, grid->grid_position, laps->completed_laps, number->car_number, position->finish_position) 
- Filter out rows where season,round, constructor_id or driver_id is null
- Remove duplicate records
- Transform values of columns sprint_name to Title Case
- Write the transformed data to silver sprints table

In [0]:
%run ../00-common/01.environment.config


In [0]:
%python
bronze_table = f"{catalog_name}.{bronze_schema_name}.sprints"
silver_table = f"{catalog_name}.{silver_schema_name}.sprints"

#Read bronze sprints table

In [0]:
%python
sprints_df = spark.read.table(bronze_table)

In [0]:
%python
display(sprints_df)
       


date,raceName,round,season,url,constructorId,driverId,grid,laps,number,points,position,positionText,status,ingestion_timestamp,source_file
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,red_bull,perez,2,17,11,8.0,1,1,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,ferrari,leclerc,1,17,16,7.0,2,2,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,red_bull,max_verstappen,3,17,1,6.0,3,3,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mercedes,russell,4,17,63,5.0,4,4,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,ferrari,sainz,5,17,55,4.0,5,5,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,aston_martin,alonso,8,17,14,3.0,6,6,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mercedes,hamilton,6,17,44,2.0,7,7,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,aston_martin,stroll,9,17,18,1.0,8,8,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,williams,albon,7,17,23,0.0,9,9,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-04-30,azerbaijan grand prix,4,2023,https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix,mclaren,piastri,11,17,81,0.0,10,10,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json


#Keep only the columns required for analytics (Drop url column)

In [0]:
%python
from pyspark.sql import functions as F
sprints_dropped_df = sprints_df.drop(F.col("url"))

#Standardise column names using snake_case (constructorId->constructor_id, driverId->driver_id, sprintName->sprint_name,positionText->finish_position_text)
#Rename columns(date->sprint_date, grid->grid_position, laps->completed_laps, number->car_number, position->finish_position)

In [0]:
%python
sprints_renamed_df = (sprints_dropped_df.withColumnsRenamed({
                        "constructorId": "constructor_id",
                        "driverId": "driver_id",
                        "raceName": "race_name",
                        "date" : "race_date",
                        "grid": "grid_position",
                        "number": "car_number",
                        "laps" : "completed_laps",
                        "position": "final_position",
                        "positionText": "final_position_text",
                       })
                      
)
        

#Filter out rows where season,round, constructor_id or driver_id is null

In [0]:
%python
sprints_valid_df = (
    sprints_renamed_df
                           .filter(
                               F.col("season").isNotNull() &
                               F.col("round").isNotNull() &
                               F.col("constructor_id").isNotNull() &
                               F.col("driver_id").isNotNull() 
                           )
                           )


#Remove duplicate records

In [0]:
%python
sprints_distinct_df = sprints_valid_df.dropDuplicates(["season","round", "race_date","constructor_id","driver_id"])

#Transform values of columns race_name to Title Case

In [0]:
%python
from pyspark.sql import functions as F
sprints_final_df = (sprints_distinct_df
                     .withColumn("race_name", F.initcap(F.col("race_name")))
                     

)

In [0]:
%python
display(sprints_final_df)

race_date,race_name,round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,final_position,final_position_text,status,ingestion_timestamp,source_file
2022-04-24,Emilia Romagna Grand Prix,4,2022,williams,albon,20,21,23,0.0,18,18,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2022.json
2022-07-10,Austrian Grand Prix,11,2022,williams,albon,11,23,23,0.0,16,16,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2022.json
2022-11-13,São Paulo Grand Prix,21,2022,williams,albon,11,12,23,0.0,20,R,Retired,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2022.json
2023-04-30,Azerbaijan Grand Prix,4,2023,williams,albon,7,17,23,0.0,9,9,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-07-02,Austrian Grand Prix,9,2023,williams,albon,11,24,23,0.0,13,13,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-07-30,Belgian Grand Prix,12,2023,williams,albon,12,11,23,0.0,12,12,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-10-08,Qatar Grand Prix,17,2023,williams,albon,17,19,23,2.0,7,7,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-10-22,United States Grand Prix,18,2023,williams,albon,8,19,23,0.0,9,9,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-11-05,São Paulo Grand Prix,20,2023,williams,albon,19,24,23,0.0,15,15,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2024-04-21,Chinese Grand Prix,5,2024,williams,albon,18,19,23,0.0,17,17,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2024.json


#Write the transformed data to silver sprints table

In [0]:
%python
(
    sprints_final_df
    .write
    .format("delta") 
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
%python
display(spark.table(silver_table))

race_date,race_name,round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,final_position,final_position_text,status,ingestion_timestamp,source_file
2022-04-24,Emilia Romagna Grand Prix,4,2022,williams,albon,20,21,23,0.0,18,18,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2022.json
2022-07-10,Austrian Grand Prix,11,2022,williams,albon,11,23,23,0.0,16,16,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2022.json
2022-11-13,São Paulo Grand Prix,21,2022,williams,albon,11,12,23,0.0,20,R,Retired,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2022.json
2023-04-30,Azerbaijan Grand Prix,4,2023,williams,albon,7,17,23,0.0,9,9,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-07-02,Austrian Grand Prix,9,2023,williams,albon,11,24,23,0.0,13,13,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-07-30,Belgian Grand Prix,12,2023,williams,albon,12,11,23,0.0,12,12,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-10-08,Qatar Grand Prix,17,2023,williams,albon,17,19,23,2.0,7,7,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-10-22,United States Grand Prix,18,2023,williams,albon,8,19,23,0.0,9,9,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2023-11-05,São Paulo Grand Prix,20,2023,williams,albon,19,24,23,0.0,15,15,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2023.json
2024-04-21,Chinese Grand Prix,5,2024,williams,albon,18,19,23,0.0,17,17,Finished,2026-06-16T14:44:14.057979Z,dbfs:/Volumes/formula1/landing/files/sprints/sprints_2024.json
